# Observations, weights and the synthetic truth

This is where the model stops being a model and starts being a *decision-support* model. So far we have a parameterised model (`part1_01`) wrapped in a PEST setup with a layered prior (`part1_02`). Here we decide three things that the rest of the series leans on:

1. **Which observations** condition the history match, and **which we deliberately hold back**.
2. **How much we believe each observation** &mdash; the weights and the measurement-noise model.
3. **What "reality" is** &mdash; the *synthetic truth* against which every forecast, posterior and emulator is scored.

None of these are mechanical. Each is a judgment call, and the honest thing to do is say so out loud and write the reasoning down. That is most of what this notebook is.

This notebook sits between [`part1_02_pstfrom_setup`](../part1_02_pstfrom_setup/dizon_pstfrom_setup.ipynb) (which built the PEST control file and drew the prior) and [`part1_04_prior_mc`](../part1_04_prior_mc/dizon_prior_mc.ipynb) (which runs the prior ensemble). The held-back observations come back to bite &mdash; in a good way &mdash; in [`part1_07_dataworth`](../part1_07_dataworth/dizon_dataworth.ipynb).

A sequencing wrinkle worth flagging up front: *choosing* the synthetic truth requires the prior Monte Carlo forecast distribution, which does not exist until `part1_04` has run. So this notebook defines the **selection criterion** and stages all the machinery; the truth is actually *picked* at the start of [`part1_05_dsi_basics`](../part1_05_dsi_basics/dizon_dsi_basics.ipynb), once the prior ensemble is in hand. We will be explicit about this where it matters.

### Admin

As in the rest of `part1`, the workspace for this notebook is created **inside this directory**. We start from the PEST template that `part1_02` produced (`pst_template/`), copy it here, and work on the copy so we never disturb the upstream notebook's outputs.

The next cell imports dependencies and asserts that the vendored `flopy`/`pyemu` are on the path. If the assertion fails, your environment is pointing at a pip-installed `pyemu` instead of the one in `dependencies/` &mdash; see the repository `README`.

In [ ]:
import os
import sys
import shutil
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import pyemu
import flopy
assert "dependencies" in flopy.__file__
assert "dependencies" in pyemu.__file__
sys.path.insert(0, "..")
import herebedragons as hbd

plt.rcParams['font.size'] = 10
pyemu.plot_utils.font = 10

**Prerequisite check.** This notebook needs the PEST template built by `part1_02`. The flush well, supply well and monitoring observations all live in that control file; if it is not there, stop and run the upstream notebook first.

In [ ]:
# the template directory produced by part1_02
org_t_d = Path("..", "part1_02_pstfrom_setup", "pst_template")
if not (org_t_d / "pest.pst").exists():
    raise Exception(
        "you need to run the '../part1_02_pstfrom_setup/dizon_pstfrom_setup.ipynb' notebook"
    )

# a working copy, created inside this notebook's directory
t_d = Path("obs_template")
if t_d.exists():
    shutil.rmtree(t_d)
shutil.copytree(org_t_d, t_d)

Load the control file as a `Pst` object and pull out the observation data. Every observation that `PstFrom` registered from the model's concentration output lands in `pst.observation_data`.

In [ ]:
pst = pyemu.Pst(str(t_d / "pest.pst"))
# re-parse the obsname tokens so oname/obsid/variable/time reflect the immutable
# obsname (part1_02 retags obgnme and overwrites the oname column; this restores oname
# to the parsed value, so "conc" selects the concentration obs robustly downstream)
pst.try_parse_name_metadata()
obs = pst.observation_data
print(f"{pst.nobs} observations registered")
obs.head()

## The timeline is the whole game

The DIZON experiment, recast as a decision problem, has three windows. They are canonical and they recur in every downstream notebook, so we draw them once here and refer back to the picture.

- **History period &mdash; day 0 to 252.** Injection runs (the injection well, `wellin`), and the flush well (`wellout`) extracts. This is the only window in which monitoring data exists, and therefore the only window that can *condition* a history match. Nothing measured after day 252 is allowed to inform the match.
- **Decision date &mdash; day 252.** The treatment/operation decision must be committed here &mdash; roughly eight weeks *before* the supply well switches on. That lead-time gap is deliberate: real decisions are made ahead of time, not on the morning the pump starts.
- **Supply period &mdash; day 308 to 728.** The supply well (`wellopt`) extracts drinking water. The forecast we care about &mdash; **peak SO&#8324; at the supply well**, the maximum over all three supply-well screens (`welopt-ly1`, `welopt-ly3`, `welopt-ly5`) and all supply-period times &mdash; lives entirely in this window. Treatment cost scales with that concentration, so the operator sizes treatment capacity to the **P95** of peak SO&#8324;: the decision is *how much treatment to build*, not whether some bright-line is crossed. (The EU drinking-water standard of 250 mg/L SO&#8324; is comfortably met here &mdash; cost, not compliance, drives this decision.)

The (728-day) tutorial timeline is an abstraction of the 854-day field experiment; we acknowledge that and move on.

In [ ]:
# model output times, from the simulation's time discretisation
sim = flopy.mf6.MFSimulation.load(sim_ws=str(t_d), load_only=['tdis'], verbosity_level=0)
perioddata = pd.DataFrame(sim.tdis.perioddata.get_data())
modeltimes = perioddata.perlen.cumsum().values
print(f"{len(modeltimes)} stress periods, simulation ends at day {modeltimes[-1]:.0f}")

The three canonical day markers:

In [ ]:
HISTORY_END   = 252   # decision date: last day usable for conditioning
SUPPLY_START  = 308   # supply well switches on
SUPPLY_END    = 728   # end of the supply period
SO4_TRIGGER   = 90    # mg/L -- illustrative supply-contract trigger (a risk lens, not the decision)

Draw the timeline figure. We reuse this exact figure (or its idea) in the part0 introduction and again wherever the windows matter.

In [ ]:
def plot_timeline(ax=None):
    """Draw the canonical DIZON decision timeline."""
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(9, 2.4))
    else:
        fig = ax.figure

    # history period band
    ax.axvspan(0, HISTORY_END, color="#cfe8cf", alpha=0.7)
    ax.text(HISTORY_END / 2, 0.72, "history period\n(monitoring data)",
            ha="center", va="center", fontsize=9)

    # lead-time gap
    ax.axvspan(HISTORY_END, SUPPLY_START, color="#eeeeee", alpha=0.9)
    ax.text((HISTORY_END + SUPPLY_START) / 2, 0.30, "lead\ntime",
            ha="center", va="center", fontsize=8, color="0.4")

    # supply period band
    ax.axvspan(SUPPLY_START, SUPPLY_END, color="#cdd8f0", alpha=0.7)
    ax.text((SUPPLY_START + SUPPLY_END) / 2, 0.72, "supply period\n(forecast: peak SO$_4$)",
            ha="center", va="center", fontsize=9)

    # decision date marker
    ax.axvline(HISTORY_END, color="crimson", lw=2)
    ax.text(HISTORY_END, 1.06, "decision date\nday 252", ha="center", va="bottom",
            fontsize=9, color="crimson")

    ax.set_xlim(0, SUPPLY_END + 10)
    ax.set_ylim(0, 1.3)
    ax.set_yticks([])
    ax.set_xlabel("time (days)")
    ax.set_xticks([0, HISTORY_END, SUPPLY_START, SUPPLY_END])
    for spine in ["left", "right", "top"]:
        ax.spines[spine].set_visible(False)
    fig.tight_layout()
    return fig, ax

_ = plot_timeline()

Keep that picture in mind: **conditioning happens entirely to the left of the red line; the forecast lives entirely in the right-hand band.** History matching never sees the supply period. That gap is exactly what makes this a forecast problem and not a curve-fitting exercise.

## The observation id scheme, once

Before we start setting weights, it pays to understand how observations are named &mdash; because we will be filtering on the name parts constantly. There are two coordinates encoded in every observation: **where** it is measured and **what species** it is.

**Monitoring sites** use the `wp1-f3` scheme:

- the **site** prefix (`wp1`, `wp2`, `wp3`, `wp4`, `pp1`, `ip2`) identifies a borehole;
- the **filter suffix** (`-f1` &hellip; `-f5`) identifies the screened interval, deepest (`-f1`) to shallowest (`-f5`).

So `wp1-f3` reads "monitoring well 1, filter depth 3". The **supply well** is the exception: its observations use **layer suffixes** (`welopt-ly1`, `welopt-ly3`, `welopt-ly5`) because that is where the forecast is read, one canonical spelling (`welopt`, not `wellopt`/`welopt` drift).

`PstFrom` stores the site as `obsid`, the species as `variable`, and the model output time as `time`. The full `obsnme` glues them together; we rarely type it by hand &mdash; we filter on the parts.

In [ ]:
# the species and sites actually present in the control file
conc = obs.loc[obs.oname == "conc"].copy()
conc['time'] = conc['time'].astype(float)
print("sites     :", sorted(conc.obsid.unique()))
print("species   :", sorted(conc.variable.unique()))
print("n times   :", conc.time.nunique(), f"(day {conc.time.min():.0f} to {conc.time.max():.0f})")

Sanity-check the site ids against the field-data file, so we never invent an observation id. Every site in the control file should be a real monitoring location in `data/obs_chem_cleaned.csv`.

In [ ]:
field = pd.read_csv(Path("..", "..", "data", "obs_chem_cleaned.csv"))
field_sites = set(field.obsid.unique())
model_sites = set(conc.obsid.unique())

missing = model_sites - field_sites
print("model sites not found in field data:", missing or "none -- all match")
print("field-only sites (no model obs)    :", field_sites - model_sites or "none")

## Conditioning species, and the cations we hold back

The model produces many species at every site. We do **not** condition on all of them. The choice of *conditioning species* is a deliberate, defensible subset:

| Species | Code | Why it conditions the history match |
|---|---|---|
| Sulfate | `so4` | The forecast species itself &mdash; the pyrite-oxidation product we ultimately care about. |
| Oxygen | `o0` | The injected oxidant. Its consumption front is the most direct fingerprint of the reaction. |
| Nitrate | `no3` | The second oxidant; together with O&#8322; it sets the redox drive on pyrite. |
| pH | `ph` | Tracks the acid/buffering balance the reaction network produces. |
| Temperature | `tmp` | A near-conservative "tracer" that constrains flow velocities (and so arrival times) almost independently of the chemistry. |

The **major cations &mdash; Ca, Mg, Na, K, Fe &mdash; are deliberately held back.** They are informative, but we are reserving them as a *dataworth* question: in `part1_07` we ask "would having measured these have actually helped the forecast?" by retraining the emulator with and without them. Holding them back now is what makes that experiment honest. We will come back to these.

One more rule: **pe (the proton-electron / redox potential variable) is never conditioned on.** It is not a measured quantity here in any reliable sense.

In [ ]:
# conditioning species (note: dissolved O2 is recorded as 'o0' in PHREEQC notation)
COND_SPECIES   = ['so4', 'o0', 'no3', 'ph', 'tmp']
# held back for the dataworth experiment in part1_07
HELD_BACK_CATS = ['ca', 'mg', 'na', 'k', 'fe', 'fe2']

present = set(conc.variable.unique())
print("conditioning species present:", [s for s in COND_SPECIES if s in present])
print("held-back cations present   :", [s for s in HELD_BACK_CATS if s in present])

## Weights: zero after the decision date

The single most important weighting decision is temporal, not chemical: **every observation after day 252 gets zero weight.** It does not matter how good a supply-period measurement would be &mdash; in the decision world of this tutorial, that data does not exist when the decision is made. Giving it weight would be cheating, and it would quietly make the history match look better than the forecast deserves.

So the rule is mechanical and absolute:

- `time <= 252` and a usable value &rarr; **non-zero weight**;
- everything else &rarr; **weight zero** (it stays in the control file so we can plot it, but it never enters phi).

We start by zeroing everything, then turn weights back on only for the conditioning species, at sites that have field data, inside the history window.

In [ ]:
# start from a clean slate: nothing is weighted
obs['time'] = obs['time'].astype(float)
obs.loc[:, "weight"] = 0.0

# candidate non-zero observations: conditioning species, in the history window,
# with a finite value (the model uses 1e30 as a no-data flag)
nz = obs.loc[
    (obs.oname == "conc")
    & (obs.variable.isin(COND_SPECIES))
    & (obs.time <= HISTORY_END)
    & (obs.obsval < 1e30)
].copy()

print(f"{len(nz)} candidate conditioning observations "
      f"(of {len(obs.loc[obs.oname=='conc'])} concentration obs total)")

Not every site carries every species in the field campaign. The supply well in particular has **no measured chemistry** &mdash; it is purely the forecast location &mdash; so it naturally drops out of the conditioning set. We restrict to sites that actually appear in the field data for each species.

In [ ]:
# (site, species) pairs that exist in the field data
field_var = field.copy()
field_var['variable'] = field_var['variable'].astype(str).str.lower()
field_pairs = set(zip(field_var.obsid, field_var.variable))

nz['pair'] = list(zip(nz.obsid, nz.variable))
nz = nz.loc[nz.pair.isin(field_pairs)].copy()
print(f"{len(nz)} conditioning observations at sites with matching field data")
print("conditioning sites:", sorted(nz.obsid.unique()))

## Species-specific measurement noise

Weight is the inverse of expected measurement noise. A flat 5% on everything (as the original prototype used) is tidy but dishonest: a pH meter and a sulfate titration do not have the same error structure, and an absolute error on pH is not a percentage of anything meaningful. So we set the standard deviation **per species**, and we justify each one in a sentence.

- **Sulfate, oxygen, nitrate (`so4`, `o0`, `no3`):** proportional error, **7%**, with a small absolute **floor** so that near-zero concentrations don't get an absurdly tiny sigma (and so don't get crushing weight). Wet-chemistry concentration measurements scale with the value; the floor protects the early, low-concentration part of each breakthrough curve.
- **pH:** an **absolute** sigma of **0.1 pH units**. pH is logarithmic &mdash; a percentage of a pH number is meaningless &mdash; and 0.1 is a fair field-measurement error for a pH probe.
- **Temperature (`tmp`):** an **absolute** sigma of **0.5 &deg;C**, a realistic accuracy for a downhole temperature logger; again, proportional error makes no physical sense for a temperature in &deg;C.

These are practitioner numbers, not derived constants. They are deliberately on the honest-to-slightly-generous side: better to under-claim our measurement precision than to over-condition the model.

In [ ]:
# species-specific noise model
PROP_FRAC   = 0.07          # proportional sigma for concentrations (so4, o0, no3)
PROP_FLOOR  = {             # absolute floor on the concentration sigma (mol/L)
    'so4': 5.0e-5,
    'o0':  2.0e-5,
    'no3': 2.0e-5,
}
ABS_SIGMA   = {             # absolute sigma where proportional makes no sense
    'ph':  0.1,             # pH units
    'tmp': 0.5,             # degrees Celsius
}

def obs_sigma(row):
    v = row.variable
    if v in ABS_SIGMA:
        return ABS_SIGMA[v]
    # proportional with a floor, for concentrations
    return max(abs(row.obsval) * PROP_FRAC, PROP_FLOOR.get(v, 0.0))

nz["standard_deviation"] = nz.apply(obs_sigma, axis=1)
nz[["obsid", "variable", "obsval", "standard_deviation"]].groupby("variable").describe()["standard_deviation"][["min", "max"]]

Turn the standard deviations into weights (`weight = 1 / sigma`) and write them back into the control file. Observations whose value is exactly zero are a special case &mdash; they get zero weight here and a zero noise realisation later, because a zero concentration carries no proportional information and an exact zero is rarely a real measurement.

In [ ]:
# weight = 1 / standard deviation
nz["weight"] = 1.0 / nz["standard_deviation"]

# exact-zero observations carry no proportional information
zero_obs = nz.loc[nz.obsval == 0].obsnme
nz.loc[nz.obsnme.isin(zero_obs), "weight"] = 0.0

# write standard_deviation and weight back into the master observation_data
obs.loc[nz.obsnme, "standard_deviation"] = nz["standard_deviation"].values
obs.loc[nz.obsnme, "weight"] = nz["weight"].values

print(f"{(obs.weight > 0).sum()} observations now carry non-zero weight")
print(f"all of them at time <= {HISTORY_END}: "
      f"{(obs.loc[obs.weight > 0, 'time'] <= HISTORY_END).all()}")

## Phi factors: no site:species group should dominate

There is a second balancing problem hiding in the weights. Sulfate concentrations are ~10&#8315;&#179; mol/L; temperatures are ~10&#185; &deg;C. Even with sensible per-species sigmas, the *number* of observations and the *magnitude* of the residuals differ wildly between groups. Left alone, the objective function (phi) would be dominated by whichever group happens to have many points or large residuals &mdash; and the history match would quietly optimise that group at the expense of the rest.

PESTPP-IES lets us rebalance this with a **phi factor file**. We group observations by **site:species** (e.g. `wp1-f3:so4`) and assign each group an equal share of the total phi. The effect: every site:species group contributes the same amount to the objective regardless of its raw residual magnitude, so **no single species or site can dominate the fit.** This is the standard "balance the components of phi" move; here it matters more than usual because the species live on such different scales.

In [ ]:
# group label: site:species
obs["obgnme"] = obs.apply(lambda r: f"{r.obsid}:{r.variable}"
                          if r.name in nz.obsnme.values else r.obgnme, axis=1)

# only the weighted groups participate in phi
weighted_groups = obs.loc[obs.weight > 0, "obgnme"].unique()
phi_factors = pd.Series(index=weighted_groups, dtype=float)
phi_factors.loc[:] = 1.0 / len(weighted_groups)

ies_phi_factor_file = "ies_phi_factors.csv"
phi_factors.to_csv(t_d / ies_phi_factor_file, header=False)
pst.pestpp_options["ies_phi_factor_file"] = ies_phi_factor_file

print(f"{len(weighted_groups)} weighted site:species groups, "
      f"each given phi share {phi_factors.iloc[0]:.4f} (sum = {phi_factors.sum():.3f})")

### Staging the noise ensemble

The history match (and, later, the emulator conditioning) needs an **observation noise ensemble** &mdash; a stack of realisations of "what the measurements could have been", drawn from the noise model we just defined. We build it here so the same noise is reused consistently downstream. Each realisation perturbs every weighted observation by a Gaussian draw at its own standard deviation; exact-zero observations stay at zero.

We will draw the full ensemble against whatever realisation count the downstream notebook uses; here we just define the helper and write a modest ensemble as a staged artifact.

In [ ]:
def make_noise_ensemble(pst_obj, num_reals, seed=0):
    """Draw an observation noise ensemble using the species-specific sigmas."""
    rng = np.random.default_rng(seed)
    od = pst_obj.observation_data
    ne = pyemu.ObservationEnsemble.from_gaussian_draw(pst_obj, num_reals=num_reals)
    nzo = od.loc[od.weight > 0].copy()
    for grp in nzo.obgnme.unique():
        rows = nzo.loc[nzo.obgnme == grp]
        sigma = rows.standard_deviation.values[0]
        offset = rng.normal(loc=0.0, scale=sigma, size=ne.shape[0])
        ne.loc[:, rows.obsnme.values] = rows.obsval.values[None, :] + offset[:, None]
    # exact-zero observations carry no noise
    z = nzo.loc[nzo.obsval == 0].obsnme.values
    ne.loc[:, z] = 0.0
    return ne

noise = make_noise_ensemble(pst, num_reals=201, seed=0)
noise = noise.loc[:, obs.loc[obs.weight > 0].obsnme]
noise.to_binary(str(t_d / "obs_noise.jcb"))
pst.pestpp_options["ies_observation_ensemble"] = "obs_noise.jcb"
print("noise ensemble:", noise.shape)

Write the staged control file. It now carries: zeroed-then-rebalanced weights, the species-specific standard deviations, the phi-factor file, and a noise ensemble. The forecast group (peak SO&#8324; at the supply well) is named in `part1_02` and is carried through untouched.

In [ ]:
pst.write(str(t_d / "pest.pst"), version=2)
print("staged control file written to", t_d / "pest.pst")

## The synthetic truth &mdash; criterion now, pick later

Everything downstream is scored against a *synthetic truth*: a single realisation we treat as "reality". Crucially, it is **not drawn at random.** A random truth often lands in the boring middle of the prior, where the prior median is already right and conditioning has nothing to correct &mdash; the tutorial would teach nothing. We choose it deliberately so that history matching has visible work to do.

**The selection criterion:** pick a prior realisation from the **upper quartile of the prior peak-SO&#8324; distribution** &mdash; concretely, the realisation whose peak supply-well SO&#8324; sits nearest the prior **87.5th percentile** (around 87&ndash;95 mg/L). "Peak supply-well SO&#8324;" is the maximum over *all three* supply-well screens (`welopt-ly1`, `welopt-ly3`, `welopt-ly5`) and all supply-period times &mdash; the canonical forecast the whole curriculum uses.

Why the upper quartile? Because then the **prior median under-predicts the truth.** Conditioning on the history-period data will pull the forecast distribution *up* toward reality &mdash; you will literally watch the posterior shift in the right direction and tighten. That is the lesson of this case: the value of data is *truer* news, not merely *better* news. A truth in the middle of the prior would hide that lesson; a truth at the very top would look like a fluke.

Here is the sequencing wrinkle made concrete. The criterion above needs the **prior Monte Carlo forecast distribution** &mdash; the spread of peak SO&#8324; across the prior ensemble. That distribution does not exist yet: the prior MC is run in [`part1_04`](../part1_04_prior_mc/dizon_prior_mc.ipynb), the *next* notebook.

We resolve this the honest way rather than forward-referencing an ensemble that hasn't been computed:

> **This notebook defines and stages the selection criterion. The truth is actually picked at the start of [`part1_05_dsi_basics`](../part1_05_dsi_basics/dizon_dsi_basics.ipynb)**, once the prior ensemble produced by `part1_04` is on disk. That keeps the sequence strictly ordered &mdash; no notebook depends on outputs from a notebook that runs later.

The selection logic is written here as a small reusable function, and `part1_05` re-applies exactly the same criterion (same max-over-screens peak, same nearest-the-87.5th-percentile rule) to the on-disk prior ensemble.

In [ ]:
SUPPLY_SCREENS = ["welopt-ly1", "welopt-ly3", "welopt-ly5"]  # all supply-well screens
TRUTH_PCTILE   = 0.875   # upper-quartile pick: nearest the prior 87.5th percentile of peak SO4


def supply_so4_cols(observation_data):
    """Column names for supply-well SO4 over the supply period, across all screens."""
    od = observation_data.copy()
    od['time'] = od['time'].astype(float)
    m = (
        (od.variable == "so4")
        & (od.obsid.astype(str).isin(SUPPLY_SCREENS))
        & (od.time >= SUPPLY_START)
        & (od.time <= SUPPLY_END)
    )
    return od.loc[m].obsnme.tolist()


def peak_supply_so4(obs_ensemble, observation_data):
    """Per-realisation peak SO4 = max over all supply screens and supply-period times (mol/L)."""
    cols = supply_so4_cols(observation_data)
    oe = pd.DataFrame(obs_ensemble)
    return oe.loc[:, cols].max(axis=1)


def pick_truth(obs_ensemble, observation_data, pctile=TRUTH_PCTILE):
    """Select the synthetic truth from the upper quartile of the prior forecast distribution.

    The truth is the realisation whose peak supply-well SO4 lies nearest the given
    percentile of the prior peak distribution (default the 87.5th -- the upper
    quartile). That puts the truth above the prior median, so conditioning visibly
    corrects the forecast upward toward it.

    Parameters
    ----------
    obs_ensemble : pyemu.ObservationEnsemble or pd.DataFrame
        Prior-MC observation ensemble (one row per realisation).
    observation_data : pd.DataFrame
        The pst.observation_data, used to locate the forecast columns.
    pctile : float
        Quantile of the prior peak distribution to target (0-1).

    Returns
    -------
    truth_idx : the realisation index chosen as the synthetic truth.
    peak : the realisation's peak supply-well SO4 (mol/L).
    """
    peak_per_real = peak_supply_so4(obs_ensemble, observation_data)
    target = peak_per_real.quantile(pctile)
    truth_idx = (peak_per_real - target).abs().idxmin()
    return truth_idx, peak_per_real.loc[truth_idx]

SO&#8324; in the model is carried in **mol/L** (matching the field-data units); we report the forecast in **mg/L** throughout the series. Sulfate's molar mass is 96.06 g/mol, so mg/L = mol/L &times; 96.06 &times; 10&sup3;. We also convert the 90 mg/L contract trigger once, for the illustrative risk lens below:

In [ ]:
SO4_MOLAR_MASS = 96.06  # g/mol

def so4_mol_to_mgl(c_mol_l):
    return c_mol_l * SO4_MOLAR_MASS * 1000.0

SO4_TRIGGER_MOL_L = SO4_TRIGGER / SO4_MOLAR_MASS / 1000.0
print(f"{SO4_TRIGGER} mg/L SO4  =  {SO4_TRIGGER_MOL_L:.4e} mol/L")

### What the pick will look like (demonstration only)

To make the criterion concrete &mdash; and to confirm the logic actually fires &mdash; we apply it here to a **prebaked** prior ensemble that the maintainer already ran for you. This is *not* the live truth selection (that happens in `part1_05`); it is a worked illustration so you can see a real number and a real curve.

The prebaked ensemble below is the prior Monte Carlo &mdash; resolved as the tracked `prebaked/prior_mc_obs_ensemble.jcb` artifact if present, otherwise a full run in the repo-root `master_priormc/` &mdash; a 201-realisation draw (a handful of which failed, as ensemble runs do, leaving ~195). At ~6 min per model run, that is roughly **20 hours of compute** the maintainer paid so you don't have to &mdash; the kind of bill that motivates the whole emulation-first approach of this series. If it is not present on your machine, the cell skips gracefully and the criterion simply waits for `part1_04`/`part1_05`.

In [ ]:
# resolve the prior MC source in the canonical order used across the series:
#   1. the tracked, thinned prebaked obs ensemble (../../prebaked/prior_mc_obs_ensemble.jcb)
#   2. a full prior MC run into the repo-root master dir (../../master_priormc)
prebaked_oe = Path("..", "..", "prebaked", "prior_mc_obs_ensemble.jcb")
master_d = Path("..", "..", "master_priormc")

if prebaked_oe.exists() and (prebaked_oe.parent / "pest.pst").exists():
    prior_pst = pyemu.Pst(str(prebaked_oe.parent / "pest.pst"))
    prior_oe = pyemu.ObservationEnsemble.from_binary(
        pst=prior_pst, filename=str(prebaked_oe))
    _src = prebaked_oe
elif (master_d / "pest.0.obs.jcb").exists():
    prior_pst = pyemu.Pst(str(master_d / "pest.pst"))
    prior_oe = pyemu.ObservationEnsemble.from_binary(
        pst=prior_pst, filename=str(master_d / "pest.0.obs.jcb"))
    _src = master_d / "pest.0.obs.jcb"
else:
    prior_oe = None

if prior_oe is not None:
    prior_pst.try_parse_name_metadata()
    truth_idx, truth_peak = pick_truth(prior_oe._df, prior_pst.observation_data)
    truth_peak_mgl = so4_mol_to_mgl(truth_peak)
    peaks_mgl = so4_mol_to_mgl(peak_supply_so4(prior_oe._df, prior_pst.observation_data))
    print(f"prior ensemble: {prior_oe.shape[0]} realisations (from {_src})")
    print(f"prior peak SO4 (max over all supply screens), mg/L:")
    print(f"  median {peaks_mgl.median():.0f}   5-95% {peaks_mgl.quantile(0.05):.0f}-"
          f"{peaks_mgl.quantile(0.95):.0f}   range {peaks_mgl.min():.0f}-{peaks_mgl.max():.0f}")
    print(f"demonstration truth: realisation {truth_idx} (nearest the "
          f"{TRUTH_PCTILE:.0%} percentile)")
    print(f"  peak supply-well SO4 = {truth_peak:.4e} mol/L = {truth_peak_mgl:.0f} mg/L")
    print(f"  prior median {peaks_mgl.median():.0f} mg/L under-predicts it -- "
          f"conditioning has work to do.")
else:
    print("prebaked prior ensemble not found -- the truth is picked in part1_05.")

And the picture: the prior spread of peak supply-well SO&#8324;, its median, and the realisation the criterion selects (in the upper quartile, above the median). This figure should make the lesson tangible &mdash; the truth sits where the prior thinks it is *unlikely*, so conditioning will have to move the forecast to find it. The dashed line is the 90 mg/L contract trigger, shown only as a risk lens (see the caption); it is not what the decision is built on.

In [ ]:
if prior_oe is not None:
    peaks = so4_mol_to_mgl(peak_supply_so4(prior_oe._df, prior_pst.observation_data))

    fig, ax = plt.subplots(1, 1, figsize=(7, 3.2))
    ax.hist(peaks, bins=30, color="0.8", edgecolor="0.5")
    ax.axvline(peaks.median(), color="0.3", lw=2,
               label=f"prior median ({peaks.median():.0f} mg/L)")
    ax.axvline(SO4_TRIGGER, color="crimson", lw=1.5, ls=":",
               label=f"{SO4_TRIGGER} mg/L contract trigger (risk lens)")
    ax.axvline(truth_peak_mgl, color="fuchsia", lw=2, ls="--",
               label=f"synthetic truth ({truth_peak_mgl:.0f} mg/L)")
    frac_over = (peaks > SO4_TRIGGER).mean()
    ax.set_xlabel("peak supply-well SO$_4$ over the supply period (mg/L)")
    ax.set_ylabel("count")
    ax.set_title(f"prior median {peaks.median():.0f} mg/L; "
                 f"P(peak > {SO4_TRIGGER} mg/L) = {frac_over:.2f}",
                 loc="left", fontsize=10)
    ax.legend(fontsize=8)
    fig.tight_layout()
else:
    print("skipping -- prior ensemble not available yet.")

That distribution &mdash; not any single threshold &mdash; is the payoff `part1_04` reads off as the *prior forecast*: a median around 82 mg/L, a 5&ndash;95% spread of roughly 72&ndash;94 mg/L. The truth we just selected sits in the upper quartile of it, near 91 mg/L. So the prior median is an **under-prediction** of reality &mdash; which is exactly what makes the lesson land. When `part1_05` conditions on the history-period data, you will see the posterior forecast shift *up* toward the truth and tighten around it: truer news, not just better news. (As a risk lens, the same picture says the prior gives the supplied water about a 15% chance of topping the 90 mg/L contract trigger; the posterior will revise that probability sharply. That is one question you can ask of the distribution &mdash; not a different analysis.)

When `part1_05` finalises the real pick, it will:

1. hold the chosen realisation **out** of the ensemble used to train the emulator (so the truth is never something the emulator has seen);
2. set the conditioning observations' `obsval` to the truth's history-period values (this is the synthetic "measured data");
3. carry the truth's supply-period peak SO&#8324; as the answer every posterior is scored against.

## What we set, and what comes next

We made four decisions and wrote each one down:

- **Conditioning species** &mdash; SO&#8324;, O&#8322; (`o0`), NO&#8323;, pH, Tmp &mdash; with the major cations **held back** for the `part1_07` dataworth experiment.
- **Weights** &mdash; zero after the day-252 decision date, non-zero only for conditioning species at field-data sites in the history window.
- **Noise** &mdash; species-specific: proportional-with-floor for concentrations, absolute 0.1 for pH and 0.5 &deg;C for temperature; turned into weights and into a reusable noise ensemble.
- **Phi factors** &mdash; equal share per site:species group, so no species or site dominates the fit.

And we **staged** the synthetic truth: defined the selection criterion (peak supply-well SO&#8324;, max over all supply screens, nearest the upper quartile &mdash; the 87.5th percentile &mdash; of the prior), wrote it as a reusable function, and showed what it produces against the prebaked prior &mdash; but deferred the actual pick to `part1_05`, after the prior Monte Carlo of `part1_04` exists.

Next: [`part1_04_prior_mc`](../part1_04_prior_mc/dizon_prior_mc.ipynb) runs the prior ensemble and reads off the prior risk.